# Exam Timetable Scheduler, Data Exploration

**ST5001CMD Artificial Intelligence** · Shweta Bhandari

This notebook explores the Kaggle *University Exam Scheduling* dataset
([kaggle.com/datasets/smrezwanulazad/exam-schedule](https://www.kaggle.com/datasets/smrezwanulazad/exam-schedule))
and the timetable my CSP solver produces from it.

After conversion the working dataset has **22 exams, 5 student
groups, 8 rooms and 12 time slots**.

Run every cell in order. Requires `pandas`, `matplotlib` and `openpyxl`:

```
pip install pandas matplotlib openpyxl
```


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.max_rows", 60)
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
})

NAVY, INDIGO, GREEN, AMBER, RED = "#1a1f36", "#818cf8", "#059669", "#d97706", "#dc2626"
PALETTE = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759", "#76b7b2", "#af7aa1"]

os.makedirs("../figures", exist_ok=True)
print("pandas", pd.__version__)

## 1. Loading the data

The solver reads a single JSON file that the converter script built from the six
Kaggle CSVs (`classrooms`, `courses`, `instructors`, `students`, `timeslots`,
`schedule`).

In [ ]:
with open("../data/exam_schedule.json", encoding="utf-8") as f:
    data = json.load(f)

print("title :", data["title"])
print("exams :", len(data["tasks"]))
print("groups:", len(data["groups"]))
print("rooms :", len(data["venues"]))
print("slots :", len(data["slots"]))

## 2. Student groups

Groups are academic cohorts (programme + year). Their sizes decide which rooms
each exam can use.

In [ ]:
groups = pd.DataFrame(
    [(g, n) for g, n in data["groups"].items()],
    columns=["group", "size"]).sort_values("size", ascending=False)

fig, ax = plt.subplots()
bars = ax.bar(groups["group"], groups["size"], color=PALETTE)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_ylabel("Students")
ax.set_title("Students per group", fontweight="bold", loc="left")
ax.set_ylim(0, groups["size"].max() * 1.18)
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.savefig("../figures/fig_group_sizes.png", bbox_inches="tight")
plt.show()

print(f"Total students: {groups['size'].sum()}")

## 3. Rooms and capacity

Every room has a fixed number of seats. If they were all the same size,
capacity would not be a real constraint.

In [ ]:
venues = pd.DataFrame(data["venues"]).sort_values("capacity", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

bars = axes[0].bar(venues["name"], venues["capacity"], color=NAVY)
axes[0].bar_label(bars, padding=3, fontsize=9)
axes[0].set_ylabel("Seats")
axes[0].set_title("Room capacity", fontweight="bold", loc="left")
axes[0].tick_params(axis="x", rotation=25)

axes[1].pie(venues["capacity"], labels=venues["name"], autopct="%1.0f%%",
            colors=PALETTE, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Share of total seating", fontweight="bold", loc="left")

plt.tight_layout()
plt.savefig("../figures/fig_rooms.png", bbox_inches="tight")
plt.show()

## 4. Exam sizes

An exam's size is the number of students sitting it. Bigger exams have fewer
rooms that fit them, and are the hardest to place.

In [ ]:
size_of = dict(zip(groups["group"], groups["size"]))

tasks = pd.DataFrame(data["tasks"])
tasks["students"] = tasks["groups"].apply(lambda gs: sum(size_of[g] for g in gs))
tasks["rooms_that_fit"] = tasks["students"].apply(
    lambda n: (venues["capacity"] >= n).sum())

fig, ax = plt.subplots(figsize=(9, 6.5))
t = tasks.sort_values("students")
colors = [RED if r == 1 else AMBER if r == 2 else INDIGO for r in t["rooms_that_fit"]]
bars = ax.barh(t["name"], t["students"], color=colors)
ax.bar_label(bars, padding=3, fontsize=8)
ax.set_xlabel("Students sitting the exam")
ax.set_title("Exam sizes (red: only 1 room fits, amber: 2)",
             fontweight="bold", loc="left")
ax.set_xlim(0, t["students"].max() * 1.15)
ax.tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig("../figures/fig_exam_sizes.png", bbox_inches="tight")
plt.show()

print(tasks["students"].describe().round(1).to_string())

## 5. Staff workload

Every teacher with more than one exam creates a potential clash: they cannot
invigilate two exams at the same time.

In [ ]:
staff_load = tasks.groupby("staff").size().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.4))
bars = ax.bar(range(len(staff_load)), staff_load.values, color=NAVY)
ax.set_xticks(range(len(staff_load)))
ax.set_xticklabels(staff_load.index, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Exams to invigilate")
ax.set_title("Staff workload", fontweight="bold", loc="left")
ax.bar_label(bars, padding=3, fontsize=9)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_ylim(0, staff_load.max() * 1.2)
plt.tight_layout()
plt.savefig("../figures/fig_staff_load.png", bbox_inches="tight")
plt.show()

print(f"{len(staff_load)} teachers, top load = {staff_load.max()} exams")

## 6. How hard is this problem?

Each exam can go in any (slot, room) pair. With this many slots and rooms,
the raw search space is astronomical: a systematic search with heuristics is
needed rather than trying combinations at random.

In [ ]:
n_slots, n_rooms, n_exams = len(data['slots']), len(data['venues']), len(data['tasks'])
positions = n_slots * n_rooms

print(f"Positions per exam        : {n_slots} slots x {n_rooms} rooms = {positions}")
print(f"Exams to place            : {n_exams}")
print(f"Raw search space          : {positions}^{n_exams} = {positions**n_exams:.3e}")
print(f"Capacity of the timetable : {positions} sittings")
print(f"Occupancy required        : {n_exams}/{positions} = {n_exams/positions:.0%}")

## 7. Running the solver

The CSP engine is imported and run. The result is analysed as a DataFrame.

In [ ]:
from engine import load_data, solve, verify, quality_score, attendance
import time

d = load_data("../data/exam_schedule.json")
stats = {"attempts": 0, "backtracks": 0}
t0 = time.time()
result = solve({}, d["tasks"], d, stats)
elapsed = time.time() - t0

cap = {v["name"]: v["capacity"] for v in d["venues"]}
schedule = pd.DataFrame([
    {"exam": n, "slot": s, "day": s.split()[0], "time": s.split()[1],
     "room": r, "capacity": cap[r],
     "students": attendance(d["_by_name"][n], d),
     "staff": d["_by_name"][n]["staff"],
     "groups": ", ".join(d["_by_name"][n]["groups"])}
    for n, (s, r) in result.items()
])
schedule["spare seats"] = schedule["capacity"] - schedule["students"]
schedule["utilisation"] = (schedule["students"] / schedule["capacity"] * 100).round(1)
order = {s: i for i, s in enumerate(d["slots"])}
schedule = schedule.sort_values("slot", key=lambda c: c.map(order)).reset_index(drop=True)

print(f"Clashes         : {len(verify(result, d))}")
print(f"Quality score   : {quality_score(result, d)}/100")
print(f"Solve time      : {elapsed:.3f}s")
print(f"Placements tried: {stats['attempts']}   Backtracks: {stats['backtracks']}")
schedule.head(10)

## 8. Is the timetable spread out?

A good exam timetable uses every available slot rather than crowding exams
into a few.

In [ ]:
per_slot = schedule.groupby("slot").size().reindex(d["slots"], fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

bars = axes[0].bar(range(len(per_slot)), per_slot.values, color=INDIGO)
axes[0].set_xticks(range(len(per_slot)))
axes[0].set_xticklabels(per_slot.index, rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("Exams")
axes[0].set_title("Exams per time slot", fontweight="bold", loc="left")
axes[0].bar_label(bars, padding=2, fontsize=8)
axes[0].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

per_day = schedule.groupby("day").size()
axes[1].pie(per_day.values, labels=per_day.index, autopct="%1.0f%%",
            colors=PALETTE, startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Share of exams by day", fontweight="bold", loc="left")

plt.tight_layout()
plt.savefig("../figures/fig_spread.png", bbox_inches="tight")
plt.show()

print(f"Slots used: {(per_slot > 0).sum()} of {len(per_slot)}")

## 9. Room utilisation

Putting a small exam in a big room is wasteful. The solver's value ordering
prefers the smallest room that fits, so utilisation should be high.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))

use = schedule.groupby("room").agg(
    exams=("exam", "size"), avg_util=("utilisation", "mean")
).sort_values("exams", ascending=False)

bars = axes[0].bar(use.index, use["exams"], color=NAVY)
axes[0].set_ylabel("Exams hosted")
axes[0].set_title("How often each room is used", fontweight="bold", loc="left")
axes[0].bar_label(bars, padding=3, fontsize=9)
axes[0].tick_params(axis="x", rotation=20)

u = schedule.sort_values("utilisation")
cols = [GREEN if x >= 80 else AMBER if x >= 60 else RED for x in u["utilisation"]]
axes[1].barh(u["exam"], u["utilisation"], color=cols)
axes[1].axvline(100, color=NAVY, linewidth=1)
axes[1].set_xlabel("Seats used (%)")
axes[1].set_title("Room utilisation per exam", fontweight="bold", loc="left")
axes[1].tick_params(axis="y", labelsize=6)

plt.tight_layout()
plt.savefig("../figures/fig_utilisation.png", bbox_inches="tight")
plt.show()

print(f"Mean utilisation : {schedule['utilisation'].mean():.1f}%")
print(f"Total spare seats: {schedule['spare seats'].sum()}")

## 10. Fairness across groups

The soft constraint penalises a group sitting two exams on the same day. A
quality score of 100 means no group has a double-exam day.

In [ ]:
rows = []
for _, r in schedule.iterrows():
    for g in r["groups"].split(", "):
        rows.append({"group": g, "day": r["day"], "exam": r["exam"]})
gd = pd.DataFrame(rows)

order = []
for s in d["slots"]:
    day = s.split()[0]
    if day not in order:
        order.append(day)

matrix = gd.pivot_table(index="group", columns="day", values="exam",
                        aggfunc="count", fill_value=0)
matrix = matrix.reindex(columns=[c for c in order if c in matrix.columns], fill_value=0)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
im = ax.imshow(matrix.values, cmap="YlOrRd", vmin=0,
               vmax=max(2, matrix.values.max()))
ax.set_xticks(range(len(matrix.columns)), matrix.columns)
ax.set_yticks(range(len(matrix.index)), matrix.index, fontsize=8)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix.values[i, j]
        ax.text(j, i, v, ha="center", va="center", fontweight="bold",
                color="white" if v >= 2 else "#333")
ax.set_title("Exams per group per day (2+ would cost quality)",
             fontweight="bold", loc="left")
plt.colorbar(im, ax=ax, shrink=0.8, label="exams")
plt.tight_layout()
plt.savefig("../figures/fig_fairness.png", bbox_inches="tight")
plt.show()

print(f"Most exams any group sits in one day: {matrix.values.max()}")

## 11. Baseline comparison

Four methods across four scenarios. This is the evaluation evidence.

| method | what it is |
|---|---|
| `full` | our system: MRV + forward checking + soft-constraint ordering |
| `no-mrv` | same solver but exams in list order, not most-constrained-first |
| `no-order` | soft constraints removed, any legal placement is accepted |
| `random` | no AI: random legal-looking assignment |


In [ ]:
from experiments import make_scenarios, solve_flex, random_baseline
import copy, time as _t

scenarios = make_scenarios("../data/exam_schedule.json")
records = []

for sname, sdata in scenarios.items():
    for method, kw in [("full",     dict(use_mrv=True,  use_ordering=True)),
                       ("no-mrv",   dict(use_mrv=False, use_ordering=True)),
                       ("no-order", dict(use_mrv=True,  use_ordering=False))]:
        s = copy.deepcopy(sdata)
        st = {"attempts": 0, "backtracks": 0}
        t0 = _t.time()
        res = solve_flex({}, s["tasks"], s, st, **kw)
        el = _t.time() - t0
        records.append({"scenario": sname, "method": method,
                        "solved": res is not None, "time": round(el, 3),
                        "tried": st["attempts"], "backtracks": st["backtracks"],
                        "clashes": len(verify(res, s)) if res else None,
                        "quality": quality_score(res, s) if res else None})
    s = copy.deepcopy(sdata)
    res = random_baseline(s)
    cl = len(verify(res, s))
    records.append({"scenario": sname, "method": "random", "solved": False,
                    "time": 0.0, "tried": 0, "backtracks": 0,
                    "clashes": cl, "quality": quality_score(res, s) if res else None})

exp = pd.DataFrame(records)
exp

### Chart: soft constraints decide quality

In [ ]:
full_only = exp[(exp["scenario"] == "full") & exp["quality"].notna()]

fig, ax = plt.subplots()
cols = [GREEN if q >= 80 else AMBER if q >= 40 else RED for q in full_only["quality"]]
bars = ax.bar(full_only["method"], full_only["quality"], color=cols)
ax.bar_label(bars, fmt="%.0f", padding=3, fontweight="bold")
ax.set_ylabel("Quality score (0-100)")
ax.set_ylim(0, 115)
ax.set_title("Schedule quality by method (full dataset)",
             fontweight="bold", loc="left")
plt.tight_layout()
plt.savefig("../figures/fig_quality_compare.png", bbox_inches="tight")
plt.show()

### Chart: the random baseline always clashes

In [ ]:
rand = exp[exp["method"] == "random"]

fig, ax = plt.subplots()
bars = ax.bar(rand["scenario"], rand["clashes"], color=RED)
ax.bar_label(bars, padding=3, fontweight="bold")
ax.set_ylabel("Constraint violations")
ax.set_title("Clashes produced by the random baseline",
             fontweight="bold", loc="left")
ax.set_ylim(0, rand["clashes"].max() * 1.2)
plt.tight_layout()
plt.savefig("../figures/fig_baseline_clashes.png", bbox_inches="tight")
plt.show()

### Chart: MRV proves impossibility faster

On solvable problems every method is quick. The difference shows on an
over-constrained problem where the system must prove that no timetable exists.
Note the logarithmic scale.

In [ ]:
imp = exp[(exp["scenario"] == "impossible") & (exp["method"] != "random")]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

bars = axes[0].bar(imp["method"], imp["time"], color=[GREEN, RED, GREEN])
axes[0].bar_label(bars, fmt="%.2fs", padding=3, fontweight="bold")
axes[0].set_ylabel("Seconds (log scale)")
axes[0].set_yscale("log")
axes[0].set_title("Time to prove impossibility", fontweight="bold", loc="left")

bars = axes[1].bar(imp["method"], imp["tried"], color=[GREEN, RED, GREEN])
axes[1].bar_label(bars, fmt="%.0f", padding=3, fontweight="bold")
axes[1].set_ylabel("Placements tried (log scale)")
axes[1].set_yscale("log")
axes[1].set_title("Search effort to prove impossibility",
                  fontweight="bold", loc="left")

plt.tight_layout()
plt.savefig("../figures/fig_mrv_benefit.png", bbox_inches="tight")
plt.show()

f = imp[imp["method"] == "full"].iloc[0]
n = imp[imp["method"] == "no-mrv"].iloc[0]
print(f"With MRV    : {f['time']:.2f}s, {f['tried']:,} placements tried")
print(f"Without MRV : {n['time']:.2f}s, {n['tried']:,} placements tried")

## 12. Summary

1. The search space is astronomical, and the random baseline confirms the
   problem is non-trivial.
2. A few big shared exams drive the difficulty. The MRV heuristic handles
   them first automatically.
3. On the solvable problem the system finds a valid timetable in a fraction
   of a second with few or no backtracks.
4. Soft constraints separate a legal timetable from a humane one.
5. MRV matters most when the answer is *no valid timetable exists*.

All figures are saved under `../figures/` for use in the report.